In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [28]:
db_path = './dataset/'

df = pd.read_csv("merged.csv")


In [29]:
print(df.columns)

Index(['Unnamed: 0', 'id_student', 'code_module', 'code_presentation', 'sum',
       'count', 'id_assessment', 'assessment_type', 'date', 'weight',
       'date_submitted', 'is_banked', 'score', 'date_registration', 'gender',
       'region', 'highest_education', 'imd_band', 'age_band',
       'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result',
       'module_presentation_length', 'study_status', 'withdrawal_status'],
      dtype='object')


In [30]:
df.shape


(1579985, 26)

In [31]:
login_freq = df.groupby('id_student')['count'].sum().reset_index()
login_freq.rename(columns={'count': 'login_frequency'}, inplace=True)

In [32]:
engagement = df.groupby('id_student')['sum'].sum().reset_index()
engagement.rename(columns={'sum': 'total_clicks'}, inplace=True)

In [33]:
regularity = df.groupby('id_student')['date'].std().reset_index()
regularity.rename(columns={'date': 'activity_std'}, inplace=True)

In [34]:
df['submission_delay'] = df['date_submitted'] - df['date']

delay = df.groupby('id_student')['submission_delay'].mean().reset_index()

In [35]:
performance = df.groupby('id_student')['score'].mean().reset_index()
performance.rename(columns={'score': 'avg_score'}, inplace=True)

In [36]:
demographics = df.groupby('id_student')[[
    'gender',
    'region',
    'highest_education',
    'imd_band',
    'age_band'
]].first().reset_index()

In [38]:
features = login_freq.merge(engagement, on='id_student')
features = features.merge(regularity, on='id_student')
features = features.merge(delay, on='id_student')
features = features.merge(performance, on='id_student')

# Optional merges
study_status = df.groupby('id_student')['study_status'].first().reset_index()
withdrawal = df.groupby('id_student')['withdrawal_status'].first().reset_index()
features = features.merge(study_status, on='id_student')
features = features.merge(withdrawal, on='id_student')
features = features.merge(demographics, on='id_student')

features.head()

,id_student,login_frequency,total_clicks,activity_std,submission_delay,avg_score,study_status,withdrawal_status,gender,region,highest_education,imd_band,age_band
0,6516,3310,13955,72.511987,-2.600000,61.800000,finished,didn't withdraw,M,Scotland,HE Qualification,80-90%,55<=
1,8462,916,1978,28.877376,-18.538462,87.307692,unfinished,late withdrawal,M,London Region,HE Qualification,30-40%,55<=
2,11391,980,4670,72.690368,-1.800000,82.000000,finished,didn't withdraw,M,East Anglian Region,HE Qualification,90-100%,55<=
3,23629,236,644,25.581397,3.500000,82.500000,finished,didn't withdraw,F,East Anglian Region,Lower Than A Level,20-30%,0-35
4,23698,2440,7280,70.811958,1.125000,73.750000,finished,didn't withdraw,F,East Anglian Region,A Level or Equivalent,50-60%,0-35


In [39]:
features.shape

(23343, 13)

In [40]:
features.to_csv("features.csv")